# Entropy-Augmented Frozen Plant DNA Language Model Embeddings for Cross-Species Polyadenylation Site Prediction

**Author:** Saleema K, Department of Computer Science, MES Keveeyam College, Valanchery
**Platform:** Kaggle Notebook (GPU T4)  ·  **Data:** Plants Genomic Benchmark (PGB, InstaDeepAI)  ·  **Model:** PlantCaduceus (Kuleshov group)

---

## Part A — Kaggle setup (do this once, about 10 minutes)

1. **Account.** Sign in at <https://www.kaggle.com>. Open your profile picture → **Settings** → **Phone verification** and verify your mobile number. GPU and Internet stay locked until you do this.
2. **Import this notebook.** Go to **Code → New Notebook**. In the notebook, choose **File → Import Notebook**, then upload this `.ipynb` file.
3. **Session options** (right-hand panel; open it with the `>|` icon if hidden):
   - **Accelerator → GPU T4 x2.** Do *not* choose P100: the Mamba kernels PlantCaduceus needs do not support it.
   - **Internet → On.** Needed to download the dataset and model from Hugging Face.
   - **Persistence → Files only** (optional). Keeps `/kaggle/working` between sessions so embeddings need not be recomputed.
4. **Run.** Click **Run All**, or run cells one by one with **Shift + Enter**. The first run takes roughly 1.5–3 hours, most of it embedding extraction. Later runs reuse the saved embeddings and take minutes.
5. **Long runs.** For an unattended full run, click **Save Version → Save & Run All (Commit)**. It keeps running after you close the browser, and all outputs (tables, figures) appear under the version's **Output** tab.
6. **Download results.** Everything is written to `/kaggle/working/results/`. Download from the **Output** panel on the right, or from the saved version.
7. **Skip the Mamba build next time.** The first run builds `mamba_ssm` and `causal_conv1d` wheels into `/kaggle/working/wheels/` (this is the slow step). After a committed run, open the version's **Output** tab → **New Dataset**, select the `wheels` folder and name it `mamba-wheels`. In later runs click **Add Input → Your Datasets → mamba-wheels**; the install cell finds the wheels and finishes in seconds. Keep the same Kaggle environment (Settings → Environment → *Pin to original environment*) so the wheels stay compatible.

**Kaggle limits to keep in mind:** about 30 GPU hours per week; 12 hours maximum per session; 20 GB of saved output.

## Part B — Inputs this notebook needs

| Input | Where it comes from | Action needed from you |
|---|---|---|
| DNA sequences + labels (poly(A) site vs. not) | Hugging Face dataset `InstaDeepAI/plant-genomic-benchmark`, configs `poly_a.*` (400 bp sequences, binary labels) | None. Downloaded automatically. |
| Pretrained DNA language model | Hugging Face `kuleshov-group/PlantCaduceus_l20` (20M params) and optionally `_l24` (40M) | None. Downloaded automatically. |
| Optional comparison model | Hugging Face `InstaDeepAI/agro-nucleotide-transformer-1b` (AgroNT, 1B params) | Set `RUN_AGRONT = True` in the config cell if you want it. |
| Hugging Face token | Not required (all resources are public) | Only if downloads are rate-limited: create a token at huggingface.co → Settings → Access Tokens, then in Kaggle **Add-ons → Secrets** add it with label `HF_TOKEN`. |
| Your settings | The **Configuration** cell below | Adjust sample sizes, species and layers if you wish. Defaults are tuned for a free T4. |

No wet-lab data, local files or paid compute are required.

## Part C — What the notebook does

| Step | Section | Output for the paper |
|---|---|---|
| 1 | Load PGB poly(A) data for 6 species, balanced subsample | Dataset table (Table 1) |
| 2 | **Information-theoretic features** (Shannon k=1–3, Rényi-2, windowed entropy profile, lagged mutual information, linguistic complexity, GC/AT skew) | Your novel feature set |
| 3 | k-mer frequency baseline (k = 3–5) | Classical baseline |
| 4 | **PlantCaduceus frozen embeddings** from several layers, with *mean* pooling and *site-window* pooling around the learned signal position | Deep representation |
| 5 | Classifiers (Logistic Regression, XGBoost) for 5 feature sets | Main results (Table 2) |
| 6 | **Layer-wise probing** | Figure: AUROC vs. layer |
| 7 | **Cross-species transfer** (train *Arabidopsis* → test 5 other species) | Table 3 + heatmap |
| 8 | Bootstrap 95% CIs and paired significance test (fusion vs. embedding) | Statistical support |
| 9 | SHAP interpretation (entropy vs. embedding contribution) | Figure: feature importance |
| 10 | t-SNE of the embedding space; positional entropy profiles | Figures |

**Research question.** Do interpretable information-theoretic sequence descriptors add complementary, local signal to frozen plant DNA language model embeddings, and does the fusion transfer better across species?

## 1. Install packages
PlantCaduceus is built on Mamba, which needs the `mamba-ssm` and `causal-conv1d` CUDA packages. The first run builds them for the T4 GPU (20–60 minutes) and keeps the wheels; attach them as a Kaggle Dataset and later runs install in seconds. If it fails, the notebook switches automatically to AgroNT (a standard transformer that needs no special kernels), so you can still complete the study.

In [ ]:
import subprocess, sys, importlib, os, glob

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
    return r.returncode == 0

pip("huggingface_hub", "xgboost", "shap", "scikit-learn", "seaborn", "logomaker")

MAMBA_OK = False
WHEEL_DIR = "/kaggle/working/wheels" if os.path.exists("/kaggle/working") else "wheels"
os.makedirs(WHEEL_DIR, exist_ok=True)
try:
    importlib.import_module("mamba_ssm"); MAMBA_OK = True
except ImportError:
    # 1) Fast path: wheels saved from an earlier run and attached as a Kaggle Dataset (Add Input)
    saved = sorted(glob.glob("/kaggle/input/**/*.whl", recursive=True))
    saved = [w for w in saved if os.path.basename(w).startswith(("causal_conv1d", "mamba_ssm"))]
    if saved:
        print("Installing saved Mamba wheels:", [os.path.basename(w) for w in saved])
        pip("--no-deps", *saved)
    else:
        # 2) Slow path (first run only): build wheels for the T4 GPU only, keep them for next time
        print("Building Mamba wheels for the T4 GPU (first run only; 20-60 minutes)...")
        os.environ.update({"TORCH_CUDA_ARCH_LIST": "7.5", "MAX_JOBS": str(os.cpu_count() or 4)})
        for pkg in ("causal-conv1d>=1.4.0", "mamba-ssm"):
            r = subprocess.run([sys.executable, "-m", "pip", "wheel", "-q", "--no-build-isolation", "--no-deps", pkg, "-w", WHEEL_DIR],
                               capture_output=True, text=True)
            if r.returncode != 0: print(r.stderr[-1500:])
        built = glob.glob(os.path.join(WHEEL_DIR, "*.whl"))
        if built: pip("--no-deps", *built)
        print("Wheels kept in", WHEEL_DIR, "-> save them as a Kaggle Dataset to skip this step next time (see Part A, step 7).")
    try:
        importlib.invalidate_caches(); importlib.import_module("mamba_ssm"); MAMBA_OK = True
    except Exception as e:
        print("mamba_ssm import failed:", e)

print("Mamba available:", MAMBA_OK)

## 2. Configuration
Change values here only. The defaults finish on a free T4 in about 2–3 hours on the first run.

In [ ]:
import os, json, math, time, random, warnings
from collections import Counter
from itertools import product
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Species available in PGB poly(A) task (dataset config suffix -> short label)
SPECIES = {
    "arabidopsis_thaliana":        "A. thaliana",
    "oryza_sativa_indica_group":   "O. sativa indica",
    "oryza_sativa_japonica_group": "O. sativa japonica",
    "medicago_truncatula":         "M. truncatula",
    "trifolium_pratense":          "T. pratense",
    "chlamydomonas_reinhardtii":   "C. reinhardtii",
}
SOURCE_SPECIES = "arabidopsis_thaliana"   # training species for cross-species transfer

N_TRAIN_PER_SPECIES = 20000   # balanced subsample of the training split (source species)
N_TEST_PER_SPECIES  = 6000    # balanced subsample of the test split (every species)
# Small "smoke test" first? Set QUICK = True to run everything in ~15 min with tiny samples.
QUICK = False
if QUICK:
    N_TRAIN_PER_SPECIES, N_TEST_PER_SPECIES = 2000, 1000

# Language models
PLANTCAD_MODELS = ["kuleshov-group/PlantCaduceus_l20"]   # add "kuleshov-group/PlantCaduceus_l24" if time permits
RUN_AGRONT      = False                                    # True = also extract AgroNT embeddings (slower, 1B params)
AGRONT_MODEL    = "InstaDeepAI/agro-nucleotide-transformer-1b"
LAYERS_TO_PROBE = "all"     # "all" or a list such as [4, 8, 12, 16, 20]
CENTER_WINDOW   = 64        # bp around the poly(A) signal for local (site-window) pooling
SITE_HALF       = 5         # bp each side of SITE for site-token pooling ("pt")
SITE            = None      # signal position in the 400 bp window; learned from A. thaliana training data in section 4b
BATCH_SIZE      = 64        # reduce to 32 if you get CUDA out-of-memory

# Paths
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else os.path.abspath("work")
RES  = os.path.join(WORK, "results");  FIG = os.path.join(RES, "figures");  EMB = os.path.join(WORK, "embeddings")
for d in (RES, FIG, EMB): os.makedirs(d, exist_ok=True)

# Optional Hugging Face token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets")
except Exception:
    pass

json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, list, dict))},
          open(os.path.join(RES, "config.json"), "w"), indent=2)
print("Results folder:", RES)

# ---------------- Figure style: one colour-blind-safe palette for every figure ----------------
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
# Okabe-Ito palette (colour-blind safe; recommended by Nature Methods "Points of view")
FS_COLORS = {"E0": "#999999", "E1": "#E69F00", "E2": "#009E73", "E3": "#0072B2", "E4": "#CC79A7", "E5": "#D55E00", "E6": "#882255", "E7": "#000000"}
FS_LABELS = {"E0": "E0 One-hot (site ±20 bp)", "E1": "E1 k-mer", "E2": "E2 Entropy", "E3": "E3 Embedding", "E4": "E3 + k-mer (E4)", "E5": "E3 + Entropy (E5)", "E6": "E0 + Entropy (E6)", "E7": "E0 + Entropy + E3 (E7)"}
# Paul Tol "bright" palette for species (sharp, colour-blind safe, distinct from the feature-set colours)
SP_COLORS = {"arabidopsis_thaliana": "#4477AA", "oryza_sativa_indica_group": "#CCBB44", "oryza_sativa_japonica_group": "#EE6677",
             "medicago_truncatula": "#228833", "trifolium_pratense": "#66CCEE", "chlamydomonas_reinhardtii": "#AA3377"}
CLASS_COLORS = {0: "#0072B2", 1: "#D55E00"}      # negative = blue, poly(A) site = vermillion
POOL_COLORS = {"mean": "#56B4E9", "ctr": "#0072B2", "pt": "#D55E00"}
POOL_NAMES = {"mean": "mean", "ctr": "site-window (64 bp)", "pt": "site-token (±5 bp)"}
fs_key = lambda tag: tag[:2]
sns.set_theme(style="ticks", context="paper")
# Journal style: Times New Roman (falls back to the metric-compatible Liberation Serif or the Times-like STIX font)
mpl.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "Nimbus Roman", "STIXGeneral", "DejaVu Serif"],
                     "mathtext.fontset": "stix",
                     "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 8,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
                     "savefig.bbox": "tight", "pdf.fonttype": 42, "svg.fonttype": "none"})
KEEP_TITLES = {"fig_logos_composition", "fig_transfer_matrix"}
def save(fig, name):
    # journal-ready export: 600 dpi PNG + vector PDF. Figure titles belong in the caption, not on the image
    if name not in KEEP_TITLES:
        for a_ in fig.axes: a_.set_title("")
        if fig._suptitle is not None: fig._suptitle.set_text("")
    fig.savefig(os.path.join(FIG, name + ".png"), dpi=600)
    fig.savefig(os.path.join(FIG, name + ".pdf"))

## 2b. Reuse an earlier run (optional, saves ~40 minutes)
If you attach the **output of your previous successful run** as an input (right panel → **Add Input → Your Work → Notebooks** → this notebook → choose the successful version), this cell copies its cached data samples, features and PlantCaduceus embeddings into the working folder. The same sequences are then used, so E0–E5 reproduce exactly and only the new feature sets (E6, E7) and analyses are added. Without such an input, the cell does nothing and everything is computed from scratch.

In [ ]:
import glob, shutil
INPUT_ROOT = os.environ.get("INPUT_ROOT", "/kaggle/input")
copied = {"data/features": 0, "embeddings": 0}
if os.path.isdir(INPUT_ROOT):
    for f in glob.glob(os.path.join(INPUT_ROOT, "**", "*.pkl"), recursive=True):
        b = os.path.basename(f)
        if b.startswith(("data_", "ent_", "kmer_")) and not os.path.exists(os.path.join(WORK, b)):
            shutil.copy(f, os.path.join(WORK, b)); copied["data/features"] += 1
    for f in glob.glob(os.path.join(INPUT_ROOT, "**", "*_site*.npz"), recursive=True):
        b = os.path.basename(f)
        if not os.path.exists(os.path.join(EMB, b)):
            shutil.copy(f, os.path.join(EMB, b)); copied["embeddings"] += 1
# Safety check: the cached samples must match this run's sizes (a quick-test version has 2,000/1,000 sequences)
_probe = os.path.join(WORK, f"data_{SOURCE_SPECIES}.pkl")
if copied["data/features"] and os.path.exists(_probe):
    n_tr = len(pd.read_pickle(_probe).get("train", []))
    if n_tr != N_TRAIN_PER_SPECIES:
        for f in glob.glob(os.path.join(WORK, "*.pkl")) + glob.glob(os.path.join(EMB, "*.npz")): os.remove(f)
        print(f"WARNING: the attached version has {n_tr} training sequences, not {N_TRAIN_PER_SPECIES}.")
        print("It is a quick-test run, so its files were NOT used. Everything will be recomputed (about 1 hour).")
        print("To use the shortcut, attach the successful FULL run instead (log shows 'train': (20000, 2)).")
        copied = {"data/features": 0, "embeddings": 0}
print("Reused from earlier run:", copied)

## 3. Load the Plants Genomic Benchmark — polyadenylation task
Each sample is a 400 bp DNA sequence; label 1 = contains a poly(A) site, 0 = does not. We draw a **class-balanced** random subsample so every species is comparable and runs stay within Kaggle limits.

In [ ]:
import re, gzip
from huggingface_hub import list_repo_files, hf_hub_download
REPO = "InstaDeepAI/plant-genomic-benchmark"
# The benchmark ships a loading script that new versions of `datasets` refuse to run,
# so we download its raw per-species FASTA files and parse them ourselves.
REPO_FILES = list_repo_files(REPO, repo_type="dataset")
POLYA_FILES = [f for f in REPO_FILES if f.startswith("poly_a/")]
print(f"{len(POLYA_FILES)} poly(A) files in the repository, e.g.", POLYA_FILES[:4])

def find_file(sp, split):
    hits = [f for f in POLYA_FILES if sp in os.path.basename(f) and split in os.path.basename(f)
            and re.search(r"\.(fa|fasta|fna)(\.gz)?$", f)]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one {split} FASTA for {sp}, found {hits}. All poly_a files: {POLYA_FILES}")
    return hits[0]

def read_fasta(path):
    # keeps each header with its own sequence, so a record can never shift onto another's label
    op = gzip.open if path.endswith(".gz") else open
    recs = []
    with op(path, "rt") as fh:
        for line in fh:
            line = line.strip()
            if not line: continue
            if line.startswith(">"): recs.append([line[1:], []])
            elif recs: recs[-1][1].append(line)
    return [h for h, _ in recs], ["".join(b) for _, b in recs]

def label_from_header(h):
    # Official rule (the benchmark's own loader): record id split on "|", second field = label
    rid = h.split()[0].replace("\\|", "|")
    parts = rid.split("|")
    if len(parts) >= 2 and parts[1].strip() in ("0", "1"):
        return int(parts[1].strip())
    m = re.search(r"label\W*([01])\b", h, re.I)
    if m: return int(m.group(1))
    toks = [t for t in re.split(r"[|\s,;:=_/-]+", h) if t]
    for t in reversed(toks):
        if t in ("0", "1"): return int(t)
    return None

def load_split(sp, split):
    path = hf_hub_download(REPO, find_file(sp, split), repo_type="dataset")
    heads, seqs = read_fasta(path)
    RAW_HEADERS[(sp, split)] = heads[:3]
    labels = [label_from_header(h) for h in heads]
    if any(l is None for l in labels) or len(set(labels)) != 2:
        script = hf_hub_download(REPO, "plant-genomic-benchmark.py", repo_type="dataset")
        hint = [l.rstrip() for l in open(script) if "label" in l.lower()][:25]
        raise ValueError(f"Could not read labels from FASTA headers of {sp}/{split}. First headers: {heads[:3]}\n"
                         "Lines mentioning 'label' in the benchmark's own loader:\n" + "\n".join(hint))
    return pd.DataFrame({"sequence": seqs, "label": labels})

def balanced_sample(df, n, seed=SEED):
    per = n // 2
    parts = [g.sample(min(per, len(g)), random_state=seed) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def clean(s):
    s = s.upper().replace("U", "T")
    return "".join(ch if ch in "ACGT" else "N" for ch in s)

RAW_HEADERS = {}
DATA = {}   # DATA[species]["train"/"test"] -> DataFrame(sequence, label)
for sp in SPECIES:
    cache = os.path.join(WORK, f"data_{sp}.pkl")
    if os.path.exists(cache):
        DATA[sp] = pd.read_pickle(cache); print("cached:", sp); continue
    out = {}
    for split in ("train", "test"):
        if sp != SOURCE_SPECIES and split == "train":
            continue                                   # target species are used for testing only
        df = load_split(sp, split)
        df["sequence"] = df["sequence"].map(clean)
        df = df[df.sequence.str.len() == df.sequence.str.len().mode()[0]]   # equal lengths for batching
        n = N_TRAIN_PER_SPECIES if split == "train" else N_TEST_PER_SPECIES
        out[split] = balanced_sample(df, n)
        print(f"  {sp} {split}: {len(df):,} sequences in file, label counts {df.label.value_counts().to_dict()}")
    DATA[sp] = out
    pd.to_pickle(out, cache)
    print(sp, {k: v.shape for k, v in out.items()})

rows = []
for sp, d in DATA.items():
    for split, df in d.items():
        rows.append({"Species": SPECIES[sp], "Split": split, "N": len(df), "Positive": int(df.label.sum()),
                     "Mean length (bp)": round(df.sequence.str.len().mean(), 1),
                     "GC %": round(100 * df.sequence.map(lambda s: (s.count("G") + s.count("C")) / max(1, len(s))).mean(), 2)})
table1 = pd.DataFrame(rows); table1.to_csv(os.path.join(RES, "table1_datasets.csv"), index=False); table1

## 4. Information-theoretic (entropy) feature extractor — *the novel component*
For each sequence we compute an interpretable descriptor vector:

| Group | Features | Why it matters for poly(A) signals |
|---|---|---|
| Global entropy | Shannon entropy of 1-, 2-, 3-mers; Rényi entropy (order 2) | Poly(A) regions are A/T-rich and low-complexity |
| **Positional entropy profile** | Shannon entropy in sliding windows (50 bp, step 25) | Captures *where* complexity drops — local signal that pooled embeddings lose |
| Centre vs. flank contrast | Entropy and AT content of the central window minus the flanks | Localises the cleavage site region |
| Lagged mutual information | MI between bases at distance 1–6 | Short-range dependency structure |
| Linguistic complexity | Observed / maximum possible distinct k-mers (k = 2–6) | Repetitiveness |
| Composition | GC content, GC skew, AT skew, A/T/G/C fractions, longest A- and T-run, AAUAAA-type hexamer counts | Known biology of plant poly(A) signals |

In [ ]:
BASES = "ACGT"
CANONICAL_PAS = ["AATAAA", "ATTAAA", "AATATA", "AATAAT", "TATAAA", "TTTTTT", "AAAAAA", "TGTA"]  # plant FUE/NUE-like motifs

def _probs(counts):
    tot = sum(counts.values())
    return np.array([c / tot for c in counts.values()]) if tot else np.array([1.0])

def shannon(seq, k=1):
    if len(seq) < k: return 0.0
    c = Counter(seq[i:i + k] for i in range(len(seq) - k + 1) if "N" not in seq[i:i + k])
    p = _probs(c); return float(-(p * np.log2(p)).sum())

def renyi2(seq, k=1):
    c = Counter(seq[i:i + k] for i in range(len(seq) - k + 1) if "N" not in seq[i:i + k])
    p = _probs(c); return float(-np.log2((p ** 2).sum()))

def mutual_info(seq, lag):
    pairs = [(seq[i], seq[i + lag]) for i in range(len(seq) - lag) if seq[i] != "N" and seq[i + lag] != "N"]
    if not pairs: return 0.0
    n = len(pairs); pj = Counter(pairs); px = Counter(a for a, _ in pairs); py = Counter(b for _, b in pairs)
    return float(sum((c / n) * math.log2((c / n) / ((px[a] / n) * (py[b] / n))) for (a, b), c in pj.items()))

def linguistic_complexity(seq, k):
    obs = len({seq[i:i + k] for i in range(len(seq) - k + 1)})
    return obs / max(1, min(4 ** k, len(seq) - k + 1))

def longest_run(seq, ch):
    best = cur = 0
    for c in seq:
        cur = cur + 1 if c == ch else 0; best = max(best, cur)
    return best

def entropy_features(seq, win=50, step=25, center=CENTER_WINDOW):
    L = len(seq); f = {}
    for k in (1, 2, 3): f[f"H{k}"] = shannon(seq, k)
    f["Renyi2"] = renyi2(seq, 1); f["Renyi2_k2"] = renyi2(seq, 2)
    # positional entropy profile (fixed number of windows for a 400 bp input)
    starts = list(range(0, max(1, L - win + 1), step))
    prof = [shannon(seq[s:s + win], 1) for s in starts]
    for i, v in enumerate(prof): f[f"Hwin_{i:02d}"] = v
    f["Hwin_min"], f["Hwin_std"], f["Hwin_argmin"] = min(prof), float(np.std(prof)), int(np.argmin(prof)) / max(1, len(prof) - 1)
    # site vs flank contrast (site = signal position learned in 4b; window centre until then)
    sc = SITE if SITE is not None else L // 2
    c0, c1 = max(0, sc - center // 2), min(L, sc + center // 2)
    mid, flank = seq[c0:c1], seq[:c0] + seq[c1:]
    at = lambda s: (s.count("A") + s.count("T")) / max(1, len(s))
    f["H_center"], f["H_flank"] = shannon(mid), shannon(flank)
    f["H_contrast"] = f["H_center"] - f["H_flank"]
    f["AT_center"], f["AT_contrast"] = at(mid), at(mid) - at(flank)
    # fine positional profile around the site: 10 bp windows, 5 bp step, from SITE-60 to SITE+40
    for i, a in enumerate(range(sc - 60, sc + 31, 5)):
        w_ = seq[max(0, a):max(0, min(L, a + 10))]
        f[f"Hfine_{i:02d}"] = shannon(w_) if w_ else 0.0
        f[f"ATfine_{i:02d}"] = (w_.count("A") + w_.count("T")) / max(1, len(w_))
    # fine windows around the site: upstream U-rich/NUE region and the site itself
    for nm, a, b in (("up40_10", sc - 40, sc - 10), ("pm10", sc - 10, sc + 10), ("dn10_40", sc + 10, sc + 40)):
        w_ = seq[max(0, a):max(0, min(L, b))]
        f[f"H_{nm}"], f[f"A_{nm}"], f[f"T_{nm}"] = shannon(w_), w_.count("A") / max(1, len(w_)), w_.count("T") / max(1, len(w_))
    for lag in range(1, 7): f[f"MI_lag{lag}"] = mutual_info(seq, lag)
    for k in range(2, 7): f[f"LC_k{k}"] = linguistic_complexity(seq, k)
    a, t, g, c = (seq.count(b) for b in BASES); n = max(1, a + t + g + c)
    f.update({"fA": a / n, "fT": t / n, "fG": g / n, "fC": c / n, "GC": (g + c) / n,
              "GC_skew": (g - c) / max(1, g + c), "AT_skew": (a - t) / max(1, a + t),
              "runA": longest_run(seq, "A"), "runT": longest_run(seq, "T")})
    for m in CANONICAL_PAS:
        f[f"motif_{m}"] = sum(1 for i in range(L - len(m) + 1) if seq[i:i + len(m)] == m)
        f[f"motif_{m}_ctr"] = sum(1 for i in range(c0, max(c0, c1 - len(m) + 1)) if seq[i:i + len(m)] == m)
    return f

def entropy_matrix(seqs):
    rows = [entropy_features(s) for s in seqs]
    return pd.DataFrame(rows).fillna(0.0)

# k-mer frequency baseline
def kmer_matrix(seqs, ks=(3, 4, 5)):
    vocab = {k: ["".join(p) for p in product(BASES, repeat=k)] for k in ks}
    idx = {k: {km: i for i, km in enumerate(v)} for k, v in vocab.items()}
    blocks = []
    for k in ks:
        M = np.zeros((len(seqs), 4 ** k), dtype=np.float32)
        for r, s in enumerate(seqs):
            for i in range(len(s) - k + 1):
                j = idx[k].get(s[i:i + k])
                if j is not None: M[r, j] += 1
            M[r] /= max(1, len(s) - k + 1)
        blocks.append(M)
    names = [f"k{k}_{km}" for k in ks for km in vocab[k]]
    return pd.DataFrame(np.hstack(blocks), columns=names)

print(pd.Series(entropy_features("ACGT" * 45 + "AATAAA" + "T" * 14 + "ACGT" * 45)).head(12))

### 4b. Data sanity check — read this before trusting any result
Answers two questions with **training data only**: do poly(A)-site sequences differ from negatives, and **where along the 400 bp window**? In the Plants Genomic Benchmark the negatives are windows from the same 3′ UTR, shifted by a few to about 50 bp, so positives and negatives contain almost the same sequence and differ mainly in *where* the cleavage signal sits. This cell finds that position (`SITE`, the point of maximum positional divergence); all local features (site-token and site-window pooling, the fine entropy profile, site–flank contrast, motif counts, logos) are centred on it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
print("Raw FASTA headers (first 3 per file):")
for k, hs in RAW_HEADERS.items(): print(" ", k, hs)
df = DATA[SOURCE_SPECIES]["train"]
pos, neg = df[df.label == 1].sequence, df[df.label == 0].sequence

# Where along the window do positives differ from negatives? Per-position Jensen-Shannon divergence
# of base composition, A. thaliana TRAINING data only (no test data used), smoothed over 5 bp.
def comp(seqs):
    L_ = len(seqs.iloc[0]); M = np.zeros((L_, 4))
    for s_ in seqs:
        for i, ch in enumerate(s_):
            j = "ACGT".find(ch)
            if j >= 0: M[i, j] += 1
    return (M + 1) / (M + 1).sum(1, keepdims=True)
P, Q = comp(pos), comp(neg); Mx = (P + Q) / 2
jsd = 0.5 * (P * np.log2(P / Mx)).sum(1) + 0.5 * (Q * np.log2(Q / Mx)).sum(1)
jsd_s = pd.Series(jsd).rolling(5, center=True, min_periods=1).mean().values
SITE = int(np.argmax(jsd_s))
print(f"Poly(A) signal position learned from training data: {SITE} of {len(jsd)} bp ({SITE - len(jsd) // 2:+d} bp from the window centre)")
json.dump({"SITE": SITE}, open(os.path.join(RES, "signal_position.json"), "w"))
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(np.arange(len(jsd)) - len(jsd) // 2, jsd_s, color=FS_COLORS["E5"], lw=1.4)
ax.axvline(SITE - len(jsd) // 2, color="black", ls="--", lw=0.8)
ax.axvspan(SITE - len(jsd) // 2 - CENTER_WINDOW // 2, SITE - len(jsd) // 2 + CENTER_WINDOW // 2, color="#F0E442", alpha=0.25, lw=0)
ax.set_xlabel("Position relative to window centre (bp)"); ax.set_ylabel("JS divergence (bits)")
ax.set_title(f"Where poly(A) and negative sequences differ ({SPECIES[SOURCE_SPECIES]}, training data)")
fig.tight_layout(); save(fig, "fig_signal_position"); plt.show()
pd.DataFrame({"position_from_centre": np.arange(len(jsd)) - len(jsd) // 2, "jsd": jsd, "jsd_smoothed": jsd_s}).to_csv(os.path.join(RES, "signal_profile.csv"), index=False)

at = lambda s, a, b: (s[a:b].count("A") + s[a:b].count("T")) / max(1, b - a)
site_w = lambda s: s[max(0, SITE - 50): SITE + 50]
rows = []
for name, f in [("A+T fraction, whole 400 bp", lambda s: at(s, 0, len(s))),
                ("A+T fraction, window centre ±50 bp", lambda s: at(s, len(s) // 2 - 50, len(s) // 2 + 50)),
                ("A+T fraction, signal ±50 bp", lambda s: at(s, SITE - 50, SITE + 50)),
                ("AATAAA/ATTAAA count, signal ±50 bp", lambda s: site_w(s).count("AATAAA") + site_w(s).count("ATTAAA")),
                ("TGTA count, signal ±50 bp", lambda s: site_w(s).count("TGTA"))]:
    a, b = pos.map(f).mean(), neg.map(f).mean()
    rows.append({"measure": name, "poly(A) sites (1)": round(a, 4), "negatives (0)": round(b, 4), "difference": round(a - b, 4)})
SANITY = pd.DataFrame(rows); print(SANITY.to_string(index=False))
def onehot(seqs, c, h=20):
    return np.array([[1.0 if s[i] == b else 0.0 for i in range(max(0, c - h), c + h + 1) for b in "ACGT"] for s in seqs])
def win_kmers(seqs, c, k=4, w=100):
    return kmer_matrix([s[max(0, c - w // 2): c + w // 2] for s in seqs], ks=(k,)).values
ck = lambda X: cross_val_score(LogisticRegression(max_iter=3000, C=0.1), X, df.label.values, cv=5, scoring="roc_auc").mean()
seqs_ = df.sequence.tolist()
print("Quick checks (logistic regression, 5-fold CV AUROC, training data only):")
print(f"  content of 100 bp around the signal (4-mer counts):        {ck(win_kmers(seqs_, SITE)):.3f}")
print(f"  exact bases at signal ±20 bp (position-specific one-hot):  {ck(onehot(seqs_, SITE)):.3f}")
print(f"  exact bases at window centre ±20 bp (control):             {ck(onehot(seqs_, len(jsd) // 2)):.3f}")
print("In this benchmark the negatives are the same 3' UTR region shifted by a few to ~50 bp (see header coordinates),")
print("so content features stay near 0.5 and only POSITION-SPECIFIC features separate the classes.")
print("The study can proceed when the signal-position one-hot is clearly above both other lines.")
SANITY.to_csv(os.path.join(RES, "data_sanity_check.csv"), index=False)

In [ ]:
from joblib import Parallel, delayed

def _cached(name, fn):
    p = os.path.join(WORK, name)
    if os.path.exists(p): return pd.read_pickle(p)
    out = fn(); out.to_pickle(p); return out

def chunked(fn, seqs, n_jobs=-1, chunk=2000):
    parts = Parallel(n_jobs=n_jobs)(delayed(fn)(seqs[i:i + chunk]) for i in range(0, len(seqs), chunk))
    return pd.concat(parts, ignore_index=True)

FEAT_ENT, FEAT_KMER, FEAT_OH = {}, {}, {}
def onehot_site(seqs, h=20):
    # position-specific baseline: the base at each position within SITE ± h (4 x 41 = 164 binary features)
    return pd.DataFrame([[1.0 if s_[i] == b else 0.0 for i in range(max(0, SITE - h), SITE + h + 1) for b in "ACGT"] for s_ in seqs])
t0 = time.time()
for sp, d in DATA.items():
    for split, df in d.items():
        key = (sp, split); seqs = df.sequence.tolist()
        FEAT_ENT[key]  = _cached(f"ent_{sp}_{split}.pkl",  lambda: chunked(entropy_matrix, seqs))
        FEAT_KMER[key] = _cached(f"kmer_{sp}_{split}.pkl", lambda: chunked(kmer_matrix, seqs))
        FEAT_OH[key] = onehot_site(seqs)
print(f"Handcrafted features done in {time.time() - t0:.0f}s;",
      "entropy dims:", FEAT_ENT[(SOURCE_SPECIES, 'train')].shape[1], "| k-mer dims:", FEAT_KMER[(SOURCE_SPECIES, 'train')].shape[1])

## 5. Frozen DNA language model embeddings (layer-wise)
For every layer we save two pooled vectors per sequence:
* **mean pooling** over all positions (the standard choice), and
* **site-window pooling** over `CENTER_WINDOW` bp around the signal position `SITE` learned in section 4b — a simple remedy for the known weakness of pooled frozen embeddings on position-sensitive tasks.

PlantCaduceus processes both DNA strands, so its hidden size is doubled; following the authors' code we average the forward and reverse-complement halves. Embeddings are stored as `.npy` files and reused on later runs.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE != "cpu" else "")

def model_tag(name): return name.split("/")[-1]

import inspect, functools
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

def _accept_new_kwargs(cls):
    # Newer transformers call tie_weights(recompute_mapping=...) etc.; older remote model code
    # (PlantCaduceus) defines tie_weights(self) with no arguments. Wrap such methods so extra
    # keyword arguments are ignored. Model weights and outputs are unchanged.
    for klass in cls.__mro__:
        if klass.__module__.startswith("transformers."):
            continue
        for name in ("tie_weights", "_init_weights", "init_weights"):
            fn = klass.__dict__.get(name)
            if fn is None or getattr(fn, "_patched", False):
                continue
            params = inspect.signature(fn).parameters.values()
            if any(p.kind == p.VAR_KEYWORD for p in params):
                continue
            allowed = {p.name for p in params}
            @functools.wraps(fn)
            def wrapper(self, *a, _fn=fn, _allowed=allowed, **kw):
                return _fn(self, *a, **{k: v for k, v in kw.items() if k in _allowed})
            wrapper._patched = True
            setattr(klass, name, wrapper)
    return cls

def load_mlm(model_name):
    cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    ref = (getattr(cfg, "auto_map", None) or {}).get("AutoModelForMaskedLM")
    if ref:   # remote code model (PlantCaduceus): patch, then load
        cls = _accept_new_kwargs(get_class_from_dynamic_module(ref, model_name))
        return cls.from_pretrained(model_name, config=cfg)
    return AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True)

@torch.inference_mode()
def extract_embeddings(model_name, seqs, kind="plantcad"):
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = load_mlm(model_name).to(DEVICE).eval()
    if kind == "agront" and DEVICE != "cpu": model = model.half()
    store = None   # {"mean": all positions, "ctr": 64 bp site window, "pt": ±SITE_HALF bp at the site}
    for b in range(0, len(seqs), BATCH_SIZE):
        batch = seqs[b:b + BATCH_SIZE]
        if kind == "plantcad":
            enc = tok(batch, return_tensors="pt", padding=False, return_attention_mask=False, return_token_type_ids=False)
            out = model(input_ids=enc["input_ids"].to(DEVICE), output_hidden_states=True)
            mask = None
        else:
            enc = tok(batch, return_tensors="pt", padding="longest")
            out = model(input_ids=enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE),
                        output_hidden_states=True)
            mask = enc["attention_mask"].to(DEVICE).unsqueeze(-1).float()
        hs = out.hidden_states[1:]                       # skip the input embedding layer
        if store is None:
            store = {k: [[] for _ in hs] for k in ("mean", "ctr", "pt")}
        for li, h in enumerate(hs):
            h = h.float()
            if kind == "plantcad":                       # average forward and reverse-complement halves
                half = h.shape[-1] // 2
                h = (h[..., :half] + torch.flip(h[..., half:], dims=[-1])) / 2   # as in the PlantCAD reference code
                T = h.shape[1]; cp = int(round(SITE * T / len(batch[0]))) if SITE is not None else T // 2
                c0, c1 = max(0, cp - CENTER_WINDOW // 2), min(T, cp + CENTER_WINDOW // 2)
                mean_v, ctr_v = h.mean(1), h[:, c0:c1].mean(1)
                pt_v = h[:, max(0, cp - SITE_HALF):min(T, cp + SITE_HALF + 1)].mean(1)
            else:                                        # 6-mer tokens: centre window in tokens
                mean_v = (h * mask).sum(1) / mask.sum(1)
                T = h.shape[1]; w = max(2, CENTER_WINDOW // 6); cp = 1 + SITE // 6 if SITE is not None else T // 2
                c0 = max(1, min(T - w, cp - w // 2))
                ctr_v = h[:, c0:c0 + w].mean(1)
                pt_v = h[:, max(1, min(T - 2, cp - 1)):max(1, min(T - 2, cp - 1)) + 2].mean(1)
            for k_, v_ in (("mean", mean_v), ("ctr", ctr_v), ("pt", pt_v)):
                store[k_][li].append(v_.cpu().numpy().astype(np.float16))
        if (b // BATCH_SIZE) % 50 == 0:
            print(f"  {model_tag(model_name)}: {b + len(batch)}/{len(seqs)}")
    del model; torch.cuda.empty_cache()
    return {k_: [np.concatenate(x) for x in v_] for k_, v_ in store.items()}

def get_embeddings(model_name, kind):
    tag = model_tag(model_name); res = {}
    for sp, d in DATA.items():
        for split, df in d.items():
            p = os.path.join(EMB, f"{tag}_{sp}_{split}_site{SITE}.npz")
            if not os.path.exists(p):
                t0 = time.time(); P_ = extract_embeddings(model_name, df.sequence.tolist(), kind)
                np.savez_compressed(p, **{f"{k_}_{i}": a for k_, v_ in P_.items() for i, a in enumerate(v_)})
                print(f"{tag} {sp} {split}: {time.time() - t0:.0f}s")
            z = np.load(p); n_layers = len([k for k in z.files if k.startswith("mean_")])
            res[(sp, split)] = {k_: [z[f"{k_}_{i}"] for i in range(n_layers)] for k_ in ("mean", "ctr", "pt") if f"{k_}_0" in z.files}
    return res

EMBS, MODEL_SRC = {}, {}
if MAMBA_OK:
    for mn in PLANTCAD_MODELS:
        EMBS[model_tag(mn)] = get_embeddings(mn, "plantcad"); MODEL_SRC[model_tag(mn)] = (mn, "plantcad")
if RUN_AGRONT or not MAMBA_OK:
    if not MAMBA_OK: print("Mamba unavailable -> using AgroNT as the language model.")
    EMBS[model_tag(AGRONT_MODEL)] = get_embeddings(AGRONT_MODEL, "agront"); MODEL_SRC[model_tag(AGRONT_MODEL)] = (AGRONT_MODEL, "agront")
MAIN_MODEL = list(EMBS)[0]
print("Models with embeddings:", list(EMBS), "| main model:", MAIN_MODEL)

## 6. Evaluation utilities
Metrics: **AUROC** (primary), **AUPRC**, accuracy, F1 and Matthews correlation coefficient (MCC). Confidence intervals come from 1,000 bootstrap resamples of the test set; the fusion-vs-embedding comparison uses a paired bootstrap test.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, matthews_corrcoef
from xgboost import XGBClassifier
import torch

def make_clf(name):
    if name == "LR":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5, n_jobs=-1))
    return XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6,
                         tree_method="hist", device="cuda" if torch.cuda.is_available() else "cpu",
                         eval_metric="logloss", random_state=SEED, n_jobs=-1)

def metrics(y, p):
    yhat = (p >= 0.5).astype(int)
    return {"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p), "ACC": accuracy_score(y, yhat),
            "F1": f1_score(y, yhat), "MCC": matthews_corrcoef(y, yhat)}

def bootstrap_ci(y, p, n=1000, seed=SEED):
    rng = np.random.default_rng(seed); s = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2: s.append(roc_auc_score(y[i], p[i]))
    return np.percentile(s, [2.5, 97.5])

def paired_bootstrap_p(y, p_a, p_b, n=1000, seed=SEED):
    # one-sided p-value for H1: AUROC(a) > AUROC(b)
    rng = np.random.default_rng(seed); worse = 0; valid = 0
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        valid += 1; worse += roc_auc_score(y[i], p_a[i]) <= roc_auc_score(y[i], p_b[i])
    return (worse + 1) / (valid + 1)

def fit_predict(clf_name, Xtr, ytr, Xte_list):
    clf = make_clf(clf_name); clf.fit(np.asarray(Xtr, dtype=np.float32), ytr)
    return clf, [clf.predict_proba(np.asarray(X, dtype=np.float32))[:, 1] for X in Xte_list]

y = {k: df.label.values for sp, d in DATA.items() for split, df in d.items() for k in [(sp, split)]}
SRC_TR, SRC_TE = (SOURCE_SPECIES, "train"), (SOURCE_SPECIES, "test")

## 7. Layer-wise probing (which layer holds the poly(A) signal?)
A logistic-regression probe is trained on each layer's embedding (mean and site-window pooling). Validation uses a held-out 20% of the *training* split, so the test set is never used to choose the layer.

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split

idx_tr, idx_va = train_test_split(np.arange(len(y[SRC_TR])), test_size=0.2, stratify=y[SRC_TR], random_state=SEED)
probe_rows = []
for mt, E in EMBS.items():
    n_layers = len(E[SRC_TR]["mean"])
    layers = range(n_layers) if LAYERS_TO_PROBE == "all" else [l - 1 for l in LAYERS_TO_PROBE if 0 < l <= n_layers]
    for pool in [k_ for k_ in ("mean", "ctr", "pt") if k_ in E[SRC_TR]]:
        for li in layers:
            X = E[SRC_TR][pool][li].astype(np.float32)
            _, (pv,) = fit_predict("LR", X[idx_tr], y[SRC_TR][idx_tr], [X[idx_va]])
            probe_rows.append({"model": mt, "pooling": pool, "layer": li + 1, "val_AUROC": roc_auc_score(y[SRC_TR][idx_va], pv)})
probe = pd.DataFrame(probe_rows); probe.to_csv(os.path.join(RES, "layer_probe.csv"), index=False)

BEST = {}   # best (pooling, layer) per model, chosen on validation only
for mt, g in probe.groupby("model"):
    r = g.loc[g.val_AUROC.idxmax()]; BEST[mt] = (r.pooling, int(r.layer) - 1)
    print(f"{mt}: best = {r.pooling} pooling, layer {int(r.layer)} (val AUROC {r.val_AUROC:.4f})")

fig, ax = plt.subplots(figsize=(7, 4))
for (mt, pool), g in probe.groupby(["model", "pooling"]):
    ax.plot(g.layer, g.val_AUROC, marker={"mean": "s", "ctr": "o", "pt": "D"}[pool], ms=4, lw=1.4, color=POOL_COLORS[pool], ls="-" if mt == MAIN_MODEL else "--", label=f"{mt} – {POOL_NAMES[pool]} pooling")
ax.set_xlabel("Layer"); ax.set_ylabel("Validation AUROC"); ax.set_title(f"Layer-wise probing ({SPECIES[SOURCE_SPECIES]})")
ax.legend(fontsize=9); fig.tight_layout(); save(fig, "fig_layer_probe"); plt.show()

## 8. Main experiment — five feature sets × two classifiers
| ID | Feature set |
|---|---|
| E0 | One-hot bases at the signal ±20 bp (position-specific baseline) |
| E1 | k-mer frequencies (classical baseline) |
| E2 | Entropy / information-theoretic features (ours) |
| E3 | Frozen DNA LM embedding, best layer (deep baseline) |
| E4 | E3 + k-mer (control: does *any* handcrafted add-on help?) |
| E5 | **E3 + entropy (proposed fusion)** |

All models are trained on *A. thaliana* and evaluated on the *A. thaliana* test split and on all other species (cross-species transfer).

In [ ]:
def feature_set(fs, key, mt=None):
    if fs == "E0_onehot": return FEAT_OH[key].values
    if fs == "E1_kmer": return FEAT_KMER[key].values
    if fs == "E2_entropy": return FEAT_ENT[key].values
    if fs == "E6_onehot+entropy": return np.hstack([FEAT_OH[key].values, FEAT_ENT[key].values])
    pool, li = BEST[mt]; emb = EMBS[mt][key][pool][li].astype(np.float32)
    if fs == "E3_embedding": return emb
    if fs == "E4_emb+kmer": return np.hstack([emb, FEAT_KMER[key].values])
    if fs == "E5_emb+entropy": return np.hstack([emb, FEAT_ENT[key].values])
    if fs == "E7_onehot+entropy+emb": return np.hstack([FEAT_OH[key].values, FEAT_ENT[key].values, emb])

FEATURE_SETS = ["E0_onehot", "E1_kmer", "E2_entropy", "E3_embedding", "E4_emb+kmer", "E5_emb+entropy", "E6_onehot+entropy", "E7_onehot+entropy+emb"]
MODEL_FREE = ("E0_onehot", "E1_kmer", "E2_entropy", "E6_onehot+entropy")
test_keys = [(sp, "test") for sp in SPECIES]
results, PRED, MODELS = [], {}, {}
for mt in EMBS:
    for fs in FEATURE_SETS:
        if fs in MODEL_FREE and mt != MAIN_MODEL: continue   # model-independent; run once
        tag = fs if fs in MODEL_FREE else f"{fs} [{mt}]"
        for cn in ("LR", "XGB"):
            t0 = time.time()
            clf, preds = fit_predict(cn, feature_set(fs, SRC_TR, mt), y[SRC_TR], [feature_set(fs, k, mt) for k in test_keys])
            MODELS[(tag, cn)] = clf
            for k, p in zip(test_keys, preds):
                PRED[(tag, cn, k[0])] = p
                lo, hi = bootstrap_ci(y[k], p, n=300 if QUICK else 1000)
                results.append({"feature_set": tag, "classifier": cn, "test_species": SPECIES[k[0]],
                                "within_species": k[0] == SOURCE_SPECIES, **metrics(y[k], p), "AUROC_CI_low": lo, "AUROC_CI_high": hi})
            print(f"{tag:35s} {cn:4s} done in {time.time() - t0:.0f}s")
RESULTS = pd.DataFrame(results); RESULTS.to_csv(os.path.join(RES, "all_results.csv"), index=False)
# per-sequence test predictions (lets ROC/PR curves be re-plotted later without re-running)
_pred = []
for (tag, cn, sp), p in PRED.items():
    _pred.append(pd.DataFrame({"feature_set": tag, "classifier": cn, "species": SPECIES[sp], "label": y[(sp, "test")], "prob": np.round(p, 5)}))
pd.concat(_pred).to_csv(os.path.join(RES, "test_predictions.csv.gz"), index=False)

table2 = RESULTS[RESULTS.within_species].round(4).sort_values(["classifier", "AUROC"], ascending=[True, False])
table2.to_csv(os.path.join(RES, "table2_within_species.csv"), index=False)
table2[["feature_set", "classifier", "AUROC", "AUROC_CI_low", "AUROC_CI_high", "AUPRC", "ACC", "F1", "MCC"]]

## 9. Cross-species transfer (train *A. thaliana* → test other species)

In [ ]:
best_clf = RESULTS[RESULTS.within_species].groupby("classifier").AUROC.mean().idxmax()
cross = RESULTS[RESULTS.classifier == best_clf].pivot_table(index="feature_set", columns="test_species", values="AUROC")
cross = cross[[SPECIES[s] for s in SPECIES]]
cross["Mean (cross-species)"] = cross.drop(columns=SPECIES[SOURCE_SPECIES]).mean(1)
cross = cross.sort_values("Mean (cross-species)", ascending=False)
cross.round(4).to_csv(os.path.join(RES, "table3_cross_species_auroc.csv"))

fig, ax = plt.subplots(figsize=(11, 0.6 * len(cross) + 2))
sns.heatmap(cross, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=0.5, linecolor="white", cbar_kws={"label": "AUROC"}, ax=ax)
ax.set_title(f"Cross-species AUROC ({best_clf}); trained on {SPECIES[SOURCE_SPECIES]}"); ax.set_xlabel(""); ax.set_ylabel("")
plt.xticks(rotation=30, ha="right"); fig.tight_layout(); save(fig, "fig_cross_species_heatmap"); plt.show()
print("Classifier used:", best_clf); cross.round(4)

## 10. Does the entropy fusion significantly beat the embedding alone?
Paired bootstrap test per species: H1 = AUROC(E5 fusion) > AUROC(E3 embedding).

In [ ]:
sig = []
for mt in EMBS:
    for cn in ("LR", "XGB"):
        for sp in SPECIES:
            a, b = PRED[(f"E5_emb+entropy [{mt}]", cn, sp)], PRED[(f"E3_embedding [{mt}]", cn, sp)]
            yy = y[(sp, "test")]
            sig.append({"model": mt, "classifier": cn, "species": SPECIES[sp],
                        "AUROC_embedding": roc_auc_score(yy, b), "AUROC_fusion": roc_auc_score(yy, a),
                        "delta": roc_auc_score(yy, a) - roc_auc_score(yy, b),
                        "p_value": paired_bootstrap_p(yy, a, b, n=300 if QUICK else 1000)})
SIG = pd.DataFrame(sig); SIG["significant_0.05"] = SIG.p_value < 0.05
SIG.round(4).to_csv(os.path.join(RES, "table4_fusion_significance.csv"), index=False); SIG.round(4)

### 10b. Does anything add to the one-hot baseline?
Paired bootstrap per species: H1 = AUROC(E6 one-hot + entropy) > AUROC(E0), and H1 = AUROC(E7 one-hot + entropy + embedding) > AUROC(E0).

In [ ]:
sig2 = []
for cn in ("LR", "XGB"):
    for sp in SPECIES:
        yy = y[(sp, "test")]; b = PRED[("E0_onehot", cn, sp)]
        for tag, name in (("E6_onehot+entropy", "E6 vs E0"), (f"E7_onehot+entropy+emb [{MAIN_MODEL}]", "E7 vs E0")):
            a = PRED[(tag, cn, sp)]
            sig2.append({"comparison": name, "classifier": cn, "species": SPECIES[sp], "AUROC_E0": roc_auc_score(yy, b),
                         "AUROC_new": roc_auc_score(yy, a), "delta": roc_auc_score(yy, a) - roc_auc_score(yy, b),
                         "p_value": paired_bootstrap_p(yy, a, b, n=300 if QUICK else 1000)})
SIG2 = pd.DataFrame(sig2); SIG2["significant_0.05"] = SIG2.p_value < 0.05
SIG2.round(4).to_csv(os.path.join(RES, "table4b_vs_onehot_significance.csv"), index=False)
for (c, cn), g in SIG2.groupby(["comparison", "classifier"]):
    print(f"{c} ({cn}): mean delta {g.delta.mean():+.4f}; significant in {int(g['significant_0.05'].sum())}/{len(g)} species")
SIG2.round(4)

## 11. Figures for the paper

In [ ]:
from sklearn.metrics import roc_curve
# (a) ROC curves, within species
fig, ax = plt.subplots(figsize=(5.5, 5))
yy = y[SRC_TE]
for tag in [t for t in RESULTS.feature_set.unique() if t.startswith(("E0", "E1", "E2", "E6")) or MAIN_MODEL in t]:
    p = PRED[(tag, best_clf, SOURCE_SPECIES)]; fpr, tpr, _ = roc_curve(yy, p)
    ax.plot(fpr, tpr, lw=1.6, color=FS_COLORS[fs_key(tag)], label=f"{tag.split(' [')[0]} (AUC={roc_auc_score(yy, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title(f"ROC – {SPECIES[SOURCE_SPECIES]} ({best_clf})"); ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); save(fig, "fig_roc_within"); plt.show()

# (b) Bar chart: mean cross-species AUROC per feature set
m = RESULTS[(RESULTS.classifier == best_clf) & (~RESULTS.within_species)].groupby("feature_set").AUROC.agg(["mean", "std"]).sort_values("mean")
fig, ax = plt.subplots(figsize=(7, 0.5 * len(m) + 1.5))
ax.barh(m.index, m["mean"], xerr=m["std"], color=[FS_COLORS[fs_key(t)] for t in m.index], capsize=3, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Mean AUROC across 5 target species"); ax.set_xlim(max(0.4, m["mean"].min() - 0.08), 1.0)
ax.set_title("Cross-species generalisation"); fig.tight_layout(); save(fig, "fig_cross_species_bar"); plt.show()

# (c) Positional entropy profiles: positive vs negative sequences, per species
win_cols = [c for c in FEAT_ENT[SRC_TE].columns if c.startswith("Hwin_") and c[5:].isdigit()]
fig, ax = plt.subplots(figsize=(8, 4.5))
for sp in SPECIES:
    F = FEAT_ENT[(sp, "test")][win_cols].values; yy = y[(sp, "test")]
    ax.plot(F[yy == 1].mean(0) - F[yy == 0].mean(0), marker="o", ms=3, lw=1.4, color=SP_COLORS[sp], label=SPECIES[sp])
    PE_ROWS = globals().setdefault("PE_ROWS", []); PE_ROWS += [{"species": SPECIES[sp], "window": i, "dH": v} for i, v in enumerate(F[yy == 1].mean(0) - F[yy == 0].mean(0))]
ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("Window index (50 bp windows, step 25 bp)")
ax.set_ylabel("ΔShannon entropy (poly(A) − non-poly(A))"); ax.set_title("Positional entropy signature of poly(A) sites")
ax.legend(fontsize=8, ncol=2); fig.tight_layout(); save(fig, "fig_positional_entropy"); plt.show()
pd.DataFrame(PE_ROWS).drop_duplicates(["species", "window"]).to_csv(os.path.join(RES, "positional_entropy.csv"), index=False)

In [ ]:
# (d) t-SNE of the best embedding layer (A. thaliana test, 3,000 points)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
pool, li = BEST[MAIN_MODEL]
X = EMBS[MAIN_MODEL][SRC_TE][pool][li].astype(np.float32); yy = y[SRC_TE]
sel = np.random.default_rng(SEED).choice(len(X), min(3000, len(X)), replace=False)
Z = TSNE(n_components=2, perplexity=40, random_state=SEED, init="pca").fit_transform(PCA(min(50, X.shape[1]), random_state=SEED).fit_transform(X[sel]))
fig, ax = plt.subplots(figsize=(5.5, 5))
for lab, name in [(0, "No poly(A) site"), (1, "Poly(A) site")]:
    ax.scatter(Z[yy[sel] == lab, 0], Z[yy[sel] == lab, 1], s=4, alpha=0.5, color=CLASS_COLORS[lab], label=name)
ax.set_title(f"t-SNE of {MAIN_MODEL} layer {li + 1} ({POOL_NAMES[pool]})"); ax.set_xticks([]); ax.set_yticks([]); ax.legend(markerscale=3)
fig.tight_layout(); save(fig, "fig_tsne"); plt.show()
pd.DataFrame({"tsne1": Z[:, 0], "tsne2": Z[:, 1], "label": yy[sel]}).to_csv(os.path.join(RES, "tsne_coords.csv"), index=False)

## 12. Interpretation with SHAP — what does the entropy block contribute?
SHAP values on the XGBoost fusion model show which features drive predictions and how much of the total attribution comes from the entropy block versus the embedding block.

In [ ]:
import shap
tag = f"E5_emb+entropy [{MAIN_MODEL}]"; clf = MODELS[(tag, "XGB")]
Xte = feature_set("E5_emb+entropy", SRC_TE, MAIN_MODEL).astype(np.float32)
n_emb = Xte.shape[1] - FEAT_ENT[SRC_TE].shape[1]
names = [f"emb_{i}" for i in range(n_emb)] + list(FEAT_ENT[SRC_TE].columns)
sub = Xte[np.random.default_rng(SEED).choice(len(Xte), min(2000, len(Xte)), replace=False)]
sv = shap.TreeExplainer(clf).shap_values(sub)
absv = np.abs(sv).mean(0)
share = pd.Series({"Embedding block": absv[:n_emb].sum(), "Entropy block": absv[n_emb:].sum()}); share /= share.sum()
share.rename("share_of_mean_abs_SHAP").to_csv(os.path.join(RES, "shap_block_share.csv"))
print("Share of mean |SHAP| attribution:\n", share.round(3))
top = pd.Series(absv, index=names).sort_values(ascending=False).head(25)
top.to_csv(os.path.join(RES, "shap_top25.csv"))

fig, ax = plt.subplots(figsize=(6.5, 7))
colors = [FS_COLORS["E2"] if not n.startswith("emb_") else FS_COLORS["E3"] for n in top.index[::-1]]
ax.barh(top.index[::-1], top.values[::-1], color=colors)
ax.set_xlabel("Mean |SHAP value|"); ax.set_title("Top-25 features (green = entropy, blue = embedding)")
fig.tight_layout(); save(fig, "fig_shap_top25"); plt.show()

ent_only = [i for i, n in enumerate(names) if not n.startswith("emb_")]
shap.summary_plot(sv[:, ent_only], sub[:, ent_only], feature_names=[names[i] for i in ent_only], max_display=15, show=False)
plt.title("Entropy features – SHAP summary"); plt.tight_layout(); save(plt.gcf(), "fig_shap_entropy_summary"); plt.show()

## 12b. Additional figures (all drawn from your own results)
Six more figures, modelled on the figure types used in the reference papers (PolyA-GLM, AgroNT, PlantCaduceus):
precision–recall curves, a metric radar, sequence logos, a species tree with transfer results, an entropy-group ablation and a poly(A)-signal perturbation test.

In [ ]:
from sklearn.metrics import precision_recall_curve
# (e) Precision-recall curves, within species
fig, ax = plt.subplots(figsize=(4.2, 3.8)); yy = y[SRC_TE]
for tag in [t for t in RESULTS.feature_set.unique() if t.startswith(("E0", "E1", "E2", "E6")) or MAIN_MODEL in t]:
    p = PRED[(tag, best_clf, SOURCE_SPECIES)]; pr, rc, _ = precision_recall_curve(yy, p)
    ax.plot(rc, pr, lw=1.6, color=FS_COLORS[fs_key(tag)], label=f"{FS_LABELS[fs_key(tag)]} (AP={average_precision_score(yy, p):.3f})")
ax.axhline(yy.mean(), color="grey", ls=":", lw=0.8)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(f"Precision–recall, {SPECIES[SOURCE_SPECIES]}"); ax.legend(loc="lower left")
fig.tight_layout(); save(fig, "fig_pr_within"); plt.show()

# (f) Radar chart of five metrics, within species
mets = ["AUROC", "AUPRC", "ACC", "F1", "MCC"]
W = RESULTS[RESULTS.within_species & (RESULTS.classifier == best_clf)]
W = W[W.feature_set.str.startswith(("E0", "E1", "E2", "E6")) | W.feature_set.str.contains(MAIN_MODEL, regex=False)]
ang = np.linspace(0, 2 * np.pi, len(mets), endpoint=False).tolist(); ang += ang[:1]
fig = plt.figure(figsize=(4.4, 4.4)); ax = fig.add_subplot(111, polar=True)
for _, r in W.iterrows():
    v = [r[m] for m in mets]; v += v[:1]; c = FS_COLORS[fs_key(r.feature_set)]
    ax.plot(ang, v, color=c, lw=1.6, label=FS_LABELS[fs_key(r.feature_set)]); ax.fill(ang, v, color=c, alpha=0.08)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(mets); lo = max(0, W[mets].values.min() - 0.05); ax.set_ylim(lo, 1)
ax.set_title(f"Five metrics, {SPECIES[SOURCE_SPECIES]} ({best_clf})", pad=18); ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.1))
save(fig, "fig_metric_radar"); plt.show()

In [ ]:
# (g) Sequence logos around the centre + positional nucleotide composition (A. thaliana and C. reinhardtii)
import logomaker
def counts_matrix(seqs, a, b, ref=200):
    M = np.zeros((b - a, 4))
    for s_ in seqs:
        for i, ch in enumerate(s_[a:b]):
            j = "ACGT".find(ch)
            if j >= 0: M[i, j] += 1
    return pd.DataFrame(M, columns=list("ACGT"), index=range(a - ref, b - ref))
LOGO_SPECIES = [SOURCE_SPECIES, "chlamydomonas_reinhardtii"]
fig, axes = plt.subplots(len(LOGO_SPECIES), 2, figsize=(10, 2.6 * len(LOGO_SPECIES)), gridspec_kw={"width_ratios": [1.6, 1]})
for row, sp in enumerate(LOGO_SPECIES):
    df = DATA[sp]["test"]; pos = df[df.label == 1].sequence.tolist(); neg = df[df.label == 0].sequence.tolist()
    info = logomaker.transform_matrix(counts_matrix(pos, max(0, SITE - 40), min(400, SITE + 20), ref=SITE), from_type="counts", to_type="information")
    logomaker.Logo(info, ax=axes[row, 0], color_scheme="classic")
    axes[row, 0].set_ylabel("Bits"); axes[row, 0].set_title(SPECIES[sp], style="italic", loc="left")
    axes[row, 0].set_xlabel("Position relative to the poly(A) signal (bp)")
    fp = counts_matrix(pos, 0, 400); fn = counts_matrix(neg, 0, 400)
    fp = fp.div(fp.sum(1), axis=0).rolling(9, center=True, min_periods=1).mean(); fn = fn.div(fn.sum(1), axis=0).rolling(9, center=True, min_periods=1).mean()
    for base, c in zip("ATGC", ["#009E73", "#D55E00", "#E69F00", "#0072B2"]):
        axes[row, 1].plot(fp.index, fp[base], color=c, lw=1.4, label=f"{base} (poly(A))")
        axes[row, 1].plot(fn.index, fn[base], color=c, lw=0.9, ls="--", alpha=0.7)
    axes[row, 1].axvline(SITE - 200, color="grey", lw=0.7, ls=":")
    axes[row, 1].set_title("Solid, poly(A) sites; dashed, negatives", loc="left", fontsize=8); axes[row, 1].set_xlabel("Position (bp)")
    axes[row, 1].set_ylabel("Frequency"); axes[row, 1].legend(ncol=1, fontsize=7, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.tight_layout(); save(fig, "fig_logos_composition"); plt.show()

In [ ]:
# (h) Species tree + cross-species AUROC (topology only; no divergence times are implied)
from matplotlib.gridspec import GridSpec
order = ["arabidopsis_thaliana", "medicago_truncatula", "trifolium_pratense",
         "oryza_sativa_indica_group", "oryza_sativa_japonica_group", "chlamydomonas_reinhardtii"]
yv = {sp: i for i, sp in enumerate(order[::-1])}
fig = plt.figure(figsize=(9, 4.0)); gs = GridSpec(1, 2, width_ratios=[1, 2.2], wspace=0.05)
axt = fig.add_subplot(gs[0]); axd = fig.add_subplot(gs[1], sharey=axt)
def seg(x0, x1, y0, y1): axt.plot([x0, x1], [y0, y1], color="black", lw=1.1)
legume = (yv["medicago_truncatula"] + yv["trifolium_pratense"]) / 2; rice = (yv["oryza_sativa_indica_group"] + yv["oryza_sativa_japonica_group"]) / 2
eudi = (yv["arabidopsis_thaliana"] + legume) / 2; angio = (eudi + rice) / 2; root = (angio + yv["chlamydomonas_reinhardtii"]) / 2
for sp in ["medicago_truncatula", "trifolium_pratense"]: seg(3, 4, yv[sp], yv[sp])
seg(3, 3, yv["medicago_truncatula"], yv["trifolium_pratense"]); seg(2, 3, legume, legume)
seg(2, 4, yv["arabidopsis_thaliana"], yv["arabidopsis_thaliana"]); seg(2, 2, yv["arabidopsis_thaliana"], legume); seg(1, 2, eudi, eudi)
for sp in ["oryza_sativa_indica_group", "oryza_sativa_japonica_group"]: seg(3.4, 4, yv[sp], yv[sp])
seg(3.4, 3.4, yv["oryza_sativa_indica_group"], yv["oryza_sativa_japonica_group"]); seg(1, 3.4, rice, rice)
seg(1, 1, eudi, rice); seg(0, 1, angio, angio); seg(0, 4, yv["chlamydomonas_reinhardtii"], yv["chlamydomonas_reinhardtii"]); seg(0, 0, angio, yv["chlamydomonas_reinhardtii"])
for sp in order:
    axt.scatter(4, yv[sp], s=36, color=SP_COLORS[sp], zorder=3, edgecolor="black", linewidth=0.4)
    axt.text(4.3, yv[sp], SPECIES[sp], va="center", fontsize=8, style="italic")
axt.text(1.08, eudi + 0.12, "Eudicots", fontsize=7, color="grey"); axt.text(1.6, rice + 0.12, "Monocots", fontsize=7, color="grey")
axt.text(1.2, yv["chlamydomonas_reinhardtii"] + 0.12, "Green alga", fontsize=7, color="grey"); axt.text(4.3, yv[SOURCE_SPECIES] + 0.35, "training species", fontsize=7, color=FS_COLORS["E5"])
axt.set_xlim(-0.2, 9.5); axt.axis("off")
R = RESULTS[(RESULTS.classifier == best_clf)]
R = R[R.feature_set.str.startswith(("E0", "E1", "E2", "E6")) | R.feature_set.str.contains(MAIN_MODEL, regex=False)]
offs = dict(zip(["E0", "E1", "E2", "E3", "E4", "E5", "E6", "E7"], np.linspace(-0.36, 0.36, 8)))
inv = {v: k for k, v in SPECIES.items()}
for _, r in R.iterrows():
    k = fs_key(r.feature_set); yy0 = yv[inv[r.test_species]] + offs[k]
    axd.errorbar(r.AUROC, yy0, xerr=[[r.AUROC - r.AUROC_CI_low], [r.AUROC_CI_high - r.AUROC]], fmt="o", ms=4, color=FS_COLORS[k], elinewidth=0.8, capsize=0)
for k in offs: axd.scatter([], [], color=FS_COLORS[k], label=FS_LABELS[k])
axd.axhspan(yv[SOURCE_SPECIES] - 0.45, yv[SOURCE_SPECIES] + 0.45, color="#F0E442", alpha=0.18, lw=0)
axd.set_xlabel("AUROC (95% bootstrap CI)"); axd.tick_params(left=False, labelleft=False); axd.spines["left"].set_visible(False)
axd.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=7, ncol=4, frameon=False); axd.set_title(f"Trained on {SPECIES[SOURCE_SPECIES]}, tested on all species ({best_clf})")
save(fig, "fig_tree_transfer"); plt.show()

In [ ]:
# (i) Entropy feature-group ablation: drop one group at a time from the fusion (E5)
ent_cols = list(FEAT_ENT[SRC_TR].columns)
GROUPS = {"Global entropy": lambda c: c in ("H1", "H2", "H3", "Renyi2", "Renyi2_k2"),
          "Positional profile": lambda c: c.startswith("Hwin_"),
          "Site–flank entropy": lambda c: c in ("H_center", "H_flank", "H_contrast") or (c.startswith("H_") and c.endswith(("up40_10", "pm10", "dn10_40"))),
          "Site–flank composition": lambda c: c in ("AT_center", "AT_contrast") or (c.startswith(("A_", "T_")) and c.endswith(("up40_10", "pm10", "dn10_40"))),
          "Mutual information": lambda c: c.startswith("MI_"),
          "Linguistic complexity": lambda c: c.startswith("LC_"),
          "Composition": lambda c: c in ("fA", "fT", "fG", "fC", "GC", "GC_skew", "AT_skew", "runA", "runT"),
          "Poly(A) motifs": lambda c: c.startswith("motif_"),
          "Fine entropy profile": lambda c: c.startswith("Hfine_"),
          "Fine A+T profile": lambda c: c.startswith("ATfine_"),
          # combined families: is information theory itself useful beyond composition?
          "ALL entropy / information measures": lambda c: c.startswith(("H", "Renyi", "MI_")),
          "ALL composition / motif measures": lambda c: c.startswith(("AT", "A_", "T_", "f", "GC", "run", "motif_"))}
pool, li = BEST[MAIN_MODEL]
def e5_without(key, drop):
    keep = [c for c in ent_cols if c not in drop]
    return np.hstack([EMBS[MAIN_MODEL][key][pool][li].astype(np.float32), FEAT_ENT[key][keep].values])
def score(drop):
    _, preds = fit_predict(best_clf, e5_without(SRC_TR, drop), y[SRC_TR], [e5_without(k, drop) for k in test_keys])
    au = [roc_auc_score(y[k], p) for k, p in zip(test_keys, preds)]
    return au[0], float(np.mean(au[1:]))
full_w, full_x = score(set())
abl = []
for g, f in GROUPS.items():
    drop = {c for c in ent_cols if f(c)}; w, x = score(drop)
    abl.append({"group": g, "n_features": len(drop), "dAUROC_within": w - full_w, "dAUROC_cross": x - full_x})
ABL = pd.DataFrame(abl).sort_values("dAUROC_cross"); ABL.round(4).to_csv(os.path.join(RES, "table_entropy_ablation.csv"), index=False)
fig, ax = plt.subplots(figsize=(5.6, 0.3 * len(ABL) + 1.3)); yy0 = np.arange(len(ABL))
ax.barh(yy0 - 0.18, ABL.dAUROC_within, height=0.36, color="#56B4E9", edgecolor="black", lw=0.4, label=f"{SPECIES[SOURCE_SPECIES]}")
ax.barh(yy0 + 0.18, ABL.dAUROC_cross, height=0.36, color=FS_COLORS["E5"], edgecolor="black", lw=0.4, label="Mean of 5 target species")
ax.set_yticks(yy0); ax.set_yticklabels([f"{g} ({n})" for g, n in zip(ABL.group, ABL.n_features)]); ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("ΔAUROC when the group is removed from E5"); ax.set_title("Entropy feature-group ablation"); ax.legend(fontsize=7)
fig.tight_layout(); save(fig, "fig_entropy_ablation"); plt.show(); ABL.round(4)

In [ ]:
# (j) Poly(A)-signal perturbation (as in PolyA-GLM, Saha et al. 2026): disrupt motifs in positive sequences
PERTURB_MOTIFS = ["AATAAA", "ATTAAA", "AATATA", "AATAAT", "TATAAA", "TGTAA"]
LEVELS = [0.0, 0.25, 0.5, 0.75, 1.0]; N_PERTURB = 300 if QUICK else 1500
rng = np.random.default_rng(SEED)
def motif_hits(s_):
    hits, i = [], 0
    while i < len(s_):
        m = next((m for m in PERTURB_MOTIFS if s_.startswith(m, i)), None)
        if m: hits.append((i, len(m))); i += len(m)
        else: i += 1
    return hits
def perturb(s_, hits, frac):
    s_ = list(s_); k = int(round(frac * len(hits)))
    for j in rng.choice(len(hits), k, replace=False) if k else []:
        st, L = hits[j]; s_[st:st + L] = list(rng.choice(list("ACGT"), L))
    return "".join(s_)
pos = DATA[SOURCE_SPECIES]["test"].query("label == 1").sequence.tolist()
cand = [(s_, motif_hits(s_)) for s_ in pos]; cand = [c for c in cand if c[1]][:N_PERTURB]
print(f"{len(cand)} positive sequences contain at least one listed motif")
PERT = []
mname, mkind = MODEL_SRC[MAIN_MODEL]
for lvl in LEVELS:
    seqs = [perturb(s_, h, lvl) for s_, h in cand]
    ent = entropy_matrix(seqs).reindex(columns=ent_cols, fill_value=0.0).values
    emb = extract_embeddings(mname, seqs, mkind)[pool][li].astype(np.float32)
    for fs, X in [("E2", ent), ("E3", emb), ("E5", np.hstack([emb, ent]))]:
        tag = {"E2": "E2_entropy", "E3": f"E3_embedding [{MAIN_MODEL}]", "E5": f"E5_emb+entropy [{MAIN_MODEL}]"}[fs]
        p = MODELS[(tag, best_clf)].predict_proba(X)[:, 1]
        PERT += [{"level": lvl, "fs": fs, "prob": v} for v in p]
PERT = pd.DataFrame(PERT); PERT.to_csv(os.path.join(RES, "perturbation_probs.csv"), index=False)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), gridspec_kw={"width_ratios": [1.6, 1]})
sns.boxplot(data=PERT, x="level", y="prob", hue="fs", palette={k: FS_COLORS[k] for k in ["E2", "E3", "E5"]}, fliersize=1, linewidth=0.7, ax=axes[0])
axes[0].set_xticklabels([f"{int(l * 100)}%" for l in LEVELS]); axes[0].set_xlabel("Share of poly(A)-signal motifs disrupted")
axes[0].set_ylabel("Predicted probability (true poly(A) sites)"); axes[0].legend(title="", loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)
for t, k in zip(axes[0].get_legend().texts, ["E2", "E3", "E5"]): t.set_text(FS_LABELS[k])
miss = PERT.assign(miss=PERT.prob < 0.5).groupby(["fs", "level"]).miss.mean().unstack(0)
for k in miss.columns: axes[1].plot([int(l * 100) for l in miss.index], 100 * miss[k], marker="o", ms=4, color=FS_COLORS[k], label=FS_LABELS[k])
axes[1].set_xlabel("Motifs disrupted (%)"); axes[1].set_ylabel("Mispredicted (%)"); axes[1].legend(fontsize=7)
fig.suptitle("Models rely on poly(A)-signal motifs: confidence falls as motifs are disrupted", fontsize=10)
fig.tight_layout(); save(fig, "fig_perturbation"); plt.show(); miss.round(3)

## 12c. Robustness experiments (answering likely reviewer questions)
1. **Train on every species, test on every species** for the two embedding-free models (E0 one-hot, E6 one-hot + descriptors). The diagonal gives *within-species* AUROC, directly comparable with published per-species results (AgroNT, PlantBiMoE); the rice rows give the *reverse direction* (monocot → dicot).
2. **Full, imbalanced test files** (about 2 negatives per positive, as released) instead of the balanced 6,000-sequence subsamples.

No PlantCaduceus embeddings are needed, so this section adds only a few minutes. Training sets are balanced subsamples of each species' official training file (same size as for *A. thaliana*); the signal position learned from *A. thaliana* training data is used unchanged.

In [ ]:
# (k) Robustness: full train-test matrix (E0, E6) + evaluation on the full imbalanced test files
ROB_FS = ["E0_onehot", "E6_onehot+entropy"]
def load_full(sp, split):
    df = load_split(sp, split); df["sequence"] = df["sequence"].map(clean)
    return df[df.sequence.str.len() == df.sequence.str.len().mode()[0]].reset_index(drop=True)
def rob_features(fs, seqs, cache_name):
    oh = onehot_site(seqs).values
    if fs == "E0_onehot": return oh
    ent = _cached(cache_name, lambda: chunked(entropy_matrix, seqs)).values
    return np.hstack([oh, ent])
t0 = time.time()
ROB_TR, ROB_FULL = {}, {}
for sp in SPECIES:
    cache = os.path.join(WORK, f"rob_{sp}.pkl")
    if os.path.exists(cache):
        ROB_TR[sp], ROB_FULL[sp] = pd.read_pickle(cache); continue
    tr = DATA[sp]["train"] if "train" in DATA[sp] else balanced_sample(load_full(sp, "train"), N_TRAIN_PER_SPECIES)
    full = load_full(sp, "test")
    ROB_TR[sp], ROB_FULL[sp] = tr, full
    pd.to_pickle((tr, full), cache)
    print(f"  {SPECIES[sp]}: train {len(tr):,} (balanced), full test {len(full):,} (positives {int(full.label.sum()):,})")
print(f"Robustness data ready in {time.time() - t0:.0f}s")

rows_m, rows_full = [], []
for fs in ROB_FS:
    Xte = {sp: rob_features(fs, DATA[sp]["test"].sequence.tolist(), f"ent_{sp}_test.pkl") for sp in SPECIES}
    Xfull = {sp: rob_features(fs, ROB_FULL[sp].sequence.tolist(), f"rob_ent_{sp}_testfull.pkl") for sp in SPECIES}
    for tr_sp in SPECIES:
        t1 = time.time()
        Xtr = rob_features(fs, ROB_TR[tr_sp].sequence.tolist(), f"ent_{tr_sp}_train.pkl" if tr_sp == SOURCE_SPECIES else f"rob_ent_{tr_sp}_train.pkl")
        clf, preds = fit_predict("XGB", Xtr, ROB_TR[tr_sp].label.values, [Xte[sp] for sp in SPECIES] + [Xfull[sp] for sp in SPECIES])
        for i, te_sp in enumerate(SPECIES):
            yy, p = y[(te_sp, "test")], preds[i]
            lo, hi = bootstrap_ci(yy, p, n=200 if QUICK else 500)
            rows_m.append({"feature_set": fs, "train_species": SPECIES[tr_sp], "test_species": SPECIES[te_sp], "AUROC": roc_auc_score(yy, p),
                           "AUROC_CI_low": lo, "AUROC_CI_high": hi, "AUPRC": average_precision_score(yy, p)})
            yf, pf = ROB_FULL[te_sp].label.values, preds[len(SPECIES) + i]
            rows_full.append({"feature_set": fs, "train_species": SPECIES[tr_sp], "test_species": SPECIES[te_sp], "n_test": len(yf),
                              "positive_fraction": yf.mean(), "AUROC_full": roc_auc_score(yf, pf), "AUPRC_full": average_precision_score(yf, pf),
                              "AUROC_balanced": roc_auc_score(yy, p)})
        print(f"{fs:20s} trained on {SPECIES[tr_sp]:20s} done in {time.time() - t1:.0f}s")
MATRIX = pd.DataFrame(rows_m); MATRIX.round(4).to_csv(os.path.join(RES, "table_transfer_matrix.csv"), index=False)
FULLTEST = pd.DataFrame(rows_full); FULLTEST.round(4).to_csv(os.path.join(RES, "table_full_test.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, fs in zip(axes, ROB_FS):
    M = MATRIX[MATRIX.feature_set == fs].pivot(index="train_species", columns="test_species", values="AUROC")
    order = [SPECIES[s] for s in SPECIES]; M = M.loc[order, order]
    sns.heatmap(M, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0.5, vmax=1.0, linewidths=0.5, linecolor="white", cbar_kws={"label": "AUROC"}, ax=ax, annot_kws={"size": 7})
    ax.set_xlabel("Test species"); ax.set_ylabel("Training species" if fs == ROB_FS[0] else "")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", style="italic"); ax.set_yticklabels(ax.get_yticklabels(), rotation=0, style="italic")
    ax.set_title({"E0_onehot": "E0 One-hot", "E6_onehot+entropy": "E6 One-hot + descriptors"}[fs])
fig.tight_layout(); save(fig, "fig_transfer_matrix"); plt.show()
diag = MATRIX[MATRIX.train_species == MATRIX.test_species].pivot(index="test_species", columns="feature_set", values="AUROC")
print("Within-species AUROC (each species trained on its own data):\n", diag.round(3))
print("\nFull imbalanced test files vs balanced subsamples (trained on own species):")
print(FULLTEST[FULLTEST.train_species == FULLTEST.test_species][["feature_set", "test_species", "positive_fraction", "AUROC_balanced", "AUROC_full", "AUPRC_full"]].round(3).to_string(index=False))

## 13. Summary and export
Prints the headline numbers you need for the Abstract and zips the results folder for download.

In [ ]:
import shutil
w = RESULTS[RESULTS.within_species & (RESULTS.classifier == best_clf)].set_index("feature_set").AUROC
x = RESULTS[(~RESULTS.within_species) & (RESULTS.classifier == best_clf)].groupby("feature_set").AUROC.mean()
summary = pd.DataFrame({"AUROC within A. thaliana": w, "Mean AUROC cross-species": x}).sort_values("Mean AUROC cross-species", ascending=False)
summary.round(4).to_csv(os.path.join(RES, "summary_headline.csv")); print(summary.round(4))
n_sig = int(SIG[(SIG.classifier == best_clf) & (SIG.model == MAIN_MODEL)]["significant_0.05"].sum())
print(f"\nFusion significantly better than embedding alone (p<0.05) in {n_sig}/{len(SPECIES)} species ({best_clf}).")
print("SHAP attribution share:", share.round(3).to_dict())
shutil.make_archive(os.path.join(WORK, "results_bundle"), "zip", RES)
print("\nDownload:", os.path.join(WORK, "results_bundle.zip"))

---
### Citation checklist for the paper
* **Data / AgroNT:** Mendoza-Revilla J., Trop E., *et al.* (2024). A foundational large language model for edible plant genomes. *Communications Biology* 7, 835. doi:10.1038/s42003-024-06465-2 (dataset licence CC BY-NC-SA 4.0).
* **PlantCaduceus:** Zhai J., Gokaslan A., *et al.* (2025). Cross-species modeling of plant genomes at single nucleotide resolution using a pretrained DNA language model. *PNAS* 122(24), e2421738122.
* **Mamba:** Gu A., Dao T. (2023). Mamba: Linear-time sequence modeling with selective state spaces. arXiv:2312.00752.
* **SHAP:** Lundberg S. M., Lee S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS*.
* **XGBoost:** Chen T., Guestrin C. (2016). XGBoost: A scalable tree boosting system. *KDD*.

### Troubleshooting
| Problem | Fix |
|---|---|
| Mamba build very slow | Normal on the first run only. Save the wheels as a Kaggle Dataset (Part A, step 7) to skip it afterwards. |
| `mamba_ssm` install fails | The notebook falls back to AgroNT automatically. To retry, restart the session (**Run → Restart & clear outputs**) and run cell 1 again. |
| `tie_weights() got an unexpected keyword argument` | Fixed by the compatibility patch in section 5 (newer `transformers` vs. PlantCaduceus code). If a similar "unexpected keyword" error appears for another method, add its name to the tuple in `_accept_new_kwargs`. |
| CUDA out of memory | Lower `BATCH_SIZE` to 32 or 16 in the Configuration cell. |
| Dataset download error / 429 | Add an `HF_TOKEN` in **Add-ons → Secrets**, then rerun the Configuration cell. |
| "Expected one ... FASTA" or "Could not read labels" error | The benchmark changed its file layout; copy the printed file list or headers into the chat and the loader can be adjusted. |
| Session ran out of time | Turn on **Persistence → Files only** and rerun: finished embeddings and features are reused from `/kaggle/working`. |
| Want a quick test first | Set `QUICK = True`, run all (about 15 min), check everything works, then set it back to `False`. |